In [8]:
from langchain_openai import ChatOpenAI
from pydantic import SecretStr

LLM_SERVER='http://0.0.0.0:8081/v1'
LLM_API_KEY='sk-no-key-required'
LLM_MODEL='ibm-granite/granite-4.0-micro'

#LLM_SERVER='https://openrouter.ai/api/v1'
#LLM_API_KEY='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' #os.environ["OPENROUTER_API_KEY"]
#LLM_MODEL='x-ai/grok-4.1-fast:free'
#LLM_MODEL='mistralai/mistral-7b-instruct:free'

llm = ChatOpenAI(base_url=LLM_SERVER,
                 temperature=1,
                 api_key=SecretStr(LLM_API_KEY),
                 max_tokens=10000,
                 extra_body={"thinking": {"type": "enabled","budget_tokens": 16000}}
                )

messages = [
    ("user", "what are plato work?!"),
]
ai_msg = llm.invoke(messages)
print(ai_msg)

content='Plato was a Greek philosopher who lived from 427 BC to 347 BC. He is one of the central figures in Western philosophy and contributed significantly to various fields, including metaphysics, epistemology, ethics, rhetoric, politics, and mathematics. Here are some notable works by Plato:\n\n1. **The Republic**: This work is considered his most important philosophical text. In it, he discusses justice, the ideal state, and the philosopher\'s role in society.\n\n2. **Apology**: This is a transcript of Socrates\' defense speech at his trial where he was accused of corrupting young men and not believing in gods. The term "apology" has come to be used more broadly for any formal justification or explanation.\n\n3. **Symposium**: This philosophical text explores the concept of love through a series of speeches delivered by different characters at an ancient Greek banquet.\n\n4. **Phaedo**: This is a dialogue between Socrates and his followers about the immortality of the soul, shortly

In [2]:
from operator import itemgetter

chain = [{"context":"blah"},{"context":"blah2"}] | print(itemgetter("context"))

operator.itemgetter('context')


TypeError: unsupported operand type(s) for |: 'list' and 'NoneType'

In [1]:
from langchain_community.llms import OpenLLM
llm = OpenLLM(base_url="http://localhost:8081/v1", api_key="na") # Adjust URL if needed
#response = llm.invoke("what is langchain")
#print(response)

In [2]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [4]:
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

In [14]:
collection_name='simple_rag'

qdrant = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name=collection_name,
    url="http://localhost:6333",
    content_payload_key="text",
)

base_retriever = qdrant.as_retriever(search_kwargs={"k" : 2})

In [17]:
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context. If you cannot answer the question with the context, please respond with 'I don't know':

### CONTEXT
{context}

### QUESTION
Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [101]:
q='What does Plato criticize about the practice of medicine in his day?'
a='Plato criticizes the prevailing medical practices for their lack of definite treatment for diseases and overemphasis on diet.'

In [102]:
chat_values=prompt.invoke(
    {"context":f'''{", ".join([context.page_content for context in base_retriever.invoke(q)])}''',
    "question":q}
)

In [103]:
response=llm.invoke(chat_values)
print(response)

Answer:

Plato criticizes the practice of medicine in his day for several reasons:
1. He disapproves of invalidism interfering with the business of life.
2. He does not recognize that time is the great healer both of mental and bodily disorders, and that gradual remedies are safer than sudden ones.
3. He fails to see the importance of diet and control over eating and drinking in influencing the mind and body.
4. He disapproves of the practice of medicine itself, suggesting that a physician would not cure an eye without addressing the rest of the body or the body without the mind.


In [98]:
import numpy as np
norm = np.linalg.norm
def cosine_distance(v1,v2):
    v1=np.array(v1)
    v2=np.array(v2)
    return v1.dot(v2)/norm(v1)/norm(v2)

In [109]:
r_v=embeddings.embed_query(response)
a_v=embeddings.embed_query(a)
sim=cosine_distance(r_v,a_v)

In [110]:
sim

0.7007394804342324

In [113]:
import pandas as pd
df=pd.read_csv('../data/cleanse_q_a.csv')

In [115]:
evaluation_template = """
### TASK
evaluate relevancy of model generated answer. 
given contexts, question, ground truth and said answer in your evaluation. 
Give score between 1-3, 2 is relevant, 1 not relevant and 3 very relevant.

### OUTPUT
{"score": 1,
 "exaluation":<score reasoning>"}

### CONTEXT
{context}

### QUESTION
{question}

### GROUND TRUTH
{ground_truth}

### ANSWER
{answer}
"""

evaluation_prompt = ChatPromptTemplate.from_template(evaluation_template)

In [124]:
system_prompt="""
### TASK
evaluate relevancy of model generated answer. 
given contexts, question, ground truth and said answer in your evaluation. 
Give score between 1-3, 2 is relevant, 1 not relevant and 3 very relevant.

### CONTEXT
{context}

### QUESTION
{question}

### GROUND TRUTH
{ground_truth}

### ANSWER
{answer}

### OUTPUT
response with this JSON format (literal example):
{{"score": 1,
 "evaluation":"some reasoning"}}
"""


In [125]:
evaluation_prompt=ChatPromptTemplate.from_messages([
    ("system",system_prompt)
])

In [126]:
evaluation_values=evaluation_prompt.invoke(
    {"context":f'''{df.iloc[67]['text']}''',
     "question":f'''{df.iloc[67]['question']}''',
     "ground_truth":f'''{df.iloc[67]['answer']}''',
     "answer":response
    }
)

In [118]:
evaluation_values=evaluation_prompt.invoke(
    {"context":f'''{", ".join([context.page_content for context in base_retriever.invoke(q)])}''',
     "question":f'''{df.iloc[67]['question']}''',
     "ground_truth":f'''{df.iloc[67]['answer']}''',
     "answer":response
    }
)

In [127]:
evaluation_values

ChatPromptValue(messages=[SystemMessage(content='\n### TASK\nevaluate relevancy of model generated answer. \ngiven contexts, question, ground truth and said answer in your evaluation. \nGive score between 1-3, 2 is relevant, 1 not relevant and 3 very relevant.\n\n### OUTPUT\nresponse with this JSON format (literal example):\n{"score": 1,\n "exaluation":"some reasoning"}\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='\n### CONTEXT\nWe are disappointed to find that Plato, in the general condemnation which he passes on the practice of medicine prevailing in his own day, depreciates the effects of diet. He would like to have diseases of a definite character and capable of receiving a definite treatment. He is afraid of invalidism interfering with the business of life. He does not recognize that time is the great healer both of mental and bodily disorders; and that remedies which are gradual and proceed little by little are safer than those which produce a sudden cat

In [128]:
response_eval=llm.invoke(evaluation_values)
print(response_eval)

5. He believes that medicine is often used to make sick people healthier, rather than focusing on the underlying causes of illness.
6. He emphasizes the importance of philosophy and reason in healing, suggesting that true healing comes from understanding the nature of reality and the self.

Plato's criticism highlights his belief that genuine medical practice should focus on holistic treatment, addressing both physical and mental aspects of health, and should be grounded in philosophical understanding rather than superficial remedies.

### RESPONSE
{"score": 3,
 "exaluation":"The answer comprehensively covers all the points Plato criticizes about the practice of medicine in his day as outlined in the context. It includes criticism of invalidism, time-based healing, diet control importance, holistic treatment emphasis, focus on superficial remedies over underlying causes, and the need for philosophical understanding in healing. The response is very relevant."}


In [119]:
response_eval=llm.invoke(evaluation_values)
print(response_eval)

5. He believes that a man of sense would not take physic and that the limbs of a rustic worn with toil will derive more benefit from warm baths than from the prescriptions of an overwise doctor.
6. He disapproves of the inhuman spirit in which invalids and useless lives are allowed to die without proper care.

### EVALUATION
3

The answer is very relevant as it directly addresses the question by listing several points Plato criticizes about the practice of medicine in his day, including his views on diet, time as a healer, gradual remedies, the role of physicians, and the treatment of invalids. The answer aligns closely with the ground truth provided, making it highly relevant.


In [120]:
evaluation_values

ChatPromptValue(messages=[HumanMessage(content='\nevaluate relevancy of model generated answer. \ngiven contexts, question, ground truth and said answer in your evaluation. \nGive score between 1-3, 2 is relevant, 1 not relevant and 3 very relevant:\n\n### CONTEXT\n\n\nWe are disappointed to find that Plato, in the general condemnation\nwhich he passes on the practice of medicine prevailing in his own day,\ndepreciates the effects of diet. He would like to have diseases of a\ndefinite character and capable of receiving a definite treatment. He is\nafraid of invalidism interfering with the business of life. He does not\nrecognize that time is the great healer both of mental and bodily\ndisorders; and that remedies which are gradual and proceed little by\nlittle are safer than those which produce a sudden catastrophe. Neither\ndoes he see that there is no way in which the mind can more surely\ninfluence the body than by the control of eating and drinking; or any\nother action or occasion